# 01 — Loan Default Data Cleaning



## 1. Reproducible setup and data lineage


In [1]:
from pathlib import Path
import hashlib
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data').exists():
    raise FileNotFoundError('Open this notebook from the Machine_Learning project folder or notebooks folder.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.eda_preprocessing import (
    cleaning_change_log,
    data_validity_audit,
    deterministic_clean,
    load_raw_data,
    post_clean_validation,
    save_table,
)

RAW_DATA = PROJECT_ROOT / 'data' / 'raw' / 'Loan_Default.csv'
CLEANED_DATA = PROJECT_ROOT / 'data' / 'processed' / 'loan_default_cleaned.csv'
CLEANING_TABLES = PROJECT_ROOT / 'outputs' / 'tables' / 'cleaning'
CLEANING_TABLES.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 100)
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw input:    {RAW_DATA}')
print(f'Clean output: {CLEANED_DATA}')


Project root: /Users/zichen/Machine_Learning
Raw input:    /Users/zichen/Machine_Learning/data/raw/Loan_Default.csv
Clean output: /Users/zichen/Machine_Learning/data/processed/loan_default_cleaned.csv


## 2. Load and validate the raw dataset


In [2]:
raw = load_raw_data(RAW_DATA)

raw_overview = pd.DataFrame({
    'metric': [
        'rows', 'columns', 'numeric columns', 'categorical columns',
        'missing target values', 'unique target values'
    ],
    'value': [
        len(raw), raw.shape[1], raw.select_dtypes(include=np.number).shape[1],
        raw.select_dtypes(exclude=np.number).shape[1], int(raw['Status'].isna().sum()),
        sorted(raw['Status'].dropna().unique().tolist())
    ]
})
save_table(raw_overview, CLEANING_TABLES / 'raw_overview.csv')
display(raw_overview)
display(raw.head())


,metric,value
0,rows,148670
1,columns,34
2,numeric columns,13
3,categorical columns,21
4,missing target values,0
5,unique target values,"[0, 1]"


,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,loan_amount,rate_of_interest,Interest_rate_spread,Upfront_charges,term,Neg_ammortization,interest_only,lump_sum_payment,property_value,construction_type,occupancy_type,Secured_by,total_units,income,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,116500,NaN,NaN,NaN,360.0,not_neg,not_int,not_lpsm,118000.0,sb,pr,home,1U,1740.0,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,206500,NaN,NaN,NaN,360.0,not_neg,not_int,lpsm,NaN,sb,pr,home,1U,4980.0,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,406500,4.56,0.2000,595.0,360.0,neg_amm,not_int,not_lpsm,508000.0,sb,pr,home,1U,9480.0,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,456500,4.25,0.6810,NaN,360.0,not_neg,not_int,not_lpsm,658000.0,sb,pr,home,1U,11880.0,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,696500,4.00,0.3042,0.0,360.0,not_neg,not_int,not_lpsm,758000.0,sb,pr,home,1U,10440.0,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [3]:
raw_profile = pd.DataFrame({
    'feature': raw.columns,
    'dtype': [str(raw[c].dtype) for c in raw.columns],
    'missing_count': [int(raw[c].isna().sum()) for c in raw.columns],
    'missing_pct': [round(float(raw[c].isna().mean() * 100), 3) for c in raw.columns],
    'unique_including_missing': [int(raw[c].nunique(dropna=False)) for c in raw.columns],
}).sort_values(['missing_pct', 'feature'], ascending=[False, True])

raw_categories = []
for column in raw.select_dtypes(exclude=np.number).columns:
    counts = raw[column].value_counts(dropna=False)
    for category, count in counts.items():
        raw_categories.append({
            'feature': column,
            'category': '<MISSING>' if pd.isna(category) else str(category),
            'count': int(count),
            'percentage': round(float(count / len(raw) * 100), 3),
        })
raw_categories = pd.DataFrame(raw_categories)

duplicate_audit = pd.DataFrame({
    'check': ['exact duplicate rows', 'duplicate IDs', 'duplicates excluding ID'],
    'count': [
        int(raw.duplicated().sum()),
        int(raw['ID'].duplicated().sum()),
        int(raw.drop(columns='ID').duplicated().sum()),
    ],
    'decision': [
        'Remove exact duplicates only if present',
        'Investigate; never drop automatically by ID',
        'Review only because different IDs may represent separate applications',
    ],
})

save_table(raw_profile, CLEANING_TABLES / 'raw_column_profile.csv')
save_table(raw_categories, CLEANING_TABLES / 'raw_category_counts.csv')
save_table(duplicate_audit, CLEANING_TABLES / 'duplicate_audit.csv')

display(raw_profile)
display(duplicate_audit)
display(raw_categories.query("feature in ['Region', 'Security_Type']"))


,feature,dtype,missing_count,missing_pct,unique_including_missing
13,Upfront_charges,float64,39642,26.664,58272
12,Interest_rate_spread,float64,36639,24.645,22517
11,rate_of_interest,float64,36439,24.510,132
33,dtir1,float64,24121,16.225,58
29,LTV,float64,15098,10.155,8485
18,property_value,float64,15098,10.155,386
23,income,float64,9150,6.155,1002
2,loan_limit,object,3344,2.249,3
4,approv_in_adv,object,908,0.611,3
27,age,object,200,0.135,8


,check,count,decision
0,exact duplicate rows,0,Remove exact duplicates only if present
1,duplicate IDs,0,Investigate; never drop automatically by ID
2,duplicates excluding ID,0,Review only because different IDs may represent separate applications


,feature,category,count,percentage
59,Region,North,74722,50.260
60,Region,south,64016,43.059
61,Region,central,8697,5.850
62,Region,North-East,1235,0.831
63,Security_Type,direct,148637,99.978
64,Security_Type,Indriect,33,0.022


## 3. Cleaning decisions


In [4]:
cleaned = deterministic_clean(raw)
changes = cleaning_change_log(raw, cleaned)
save_table(changes, CLEANING_TABLES / 'cleaning_change_log.csv')
display(changes)


,cleaning_action,items_changed,reason,risk_or_limitation
0,Standardise column names to snake_case and correct neg_amortization spelling,13,"Consistent, reproducible feature naming","Names change, but values do not"
1,Strip leading/trailing whitespace from categorical values,0,Prevent accidental duplicate categories,Internal whitespace is intentionally preserved
2,Standardise south/central region capitalisation,72713,Merge labels that differ only by case,Assumes case has no semantic meaning
3,Correct Indriect security label to Indirect,33,Treat apparent spelling error as one category,Assumption should be confirmed against source documentation
4,Remove exact duplicate rows,0,Prevent repeated identical observations,No deduplication is performed on ID alone


## 4. Validity and suspicious-value audit


In [5]:
validity = data_validity_audit(cleaned)
save_table(validity, CLEANING_TABLES / 'validity_audit.csv')
display(validity)


,rule,severity,flagged_count,flagged_pct,recommended_action
0,numeric_infinite,Impossible,0,0.000,Investigate source or set to missing only with evidence
1,loan_amount_non_positive,Impossible,0,0.000,Investigate source; loan amount should be positive
2,term_non_positive,Impossible,0,0.000,Investigate source; term should be positive
3,credit_score_outside_300_900,Potentially impossible,0,0.000,Confirm scoring scale with source
4,negative_income,Potentially impossible,0,0.000,Confirm units and definition
5,zero_income,Suspicious but plausible,1260,0.848,Retain; examine missingness/zero semantics
6,non_positive_interest_rate,Suspicious,1,0.001,Retain pending domain verification
7,non_positive_property_value,Potentially impossible,0,0.000,Confirm source; do not repair automatically
8,ltv_above_200_pct,Extreme / possible data issue,17,0.011,Investigate individually; do not remove by IQR alone
9,dti_outside_0_100,Potentially impossible,0,0.000,Confirm ratio definition and range


## 5. Post-cleaning validation gate


In [6]:
validation = post_clean_validation(raw, cleaned)
save_table(validation, CLEANING_TABLES / 'post_clean_validation.csv')
display(validation)

failed = validation.loc[~validation['passed']]
assert failed.empty, f'Cleaning validation failed:\n{failed.to_string(index=False)}'
print('All post-cleaning validation checks passed.')


,validation_check,passed,expected,observed
0,Expected row count after exact deduplication,True,148670,148670
1,Column names are unique,True,True,True
2,Target has no missing values,True,0,0
3,Target remains binary 0/1,True,"{0, 1}","[0, 1]"
4,Identifier has no missing values,True,0,0
5,Identifier remains unique,True,True,True
6,No exact duplicates remain,True,0,0
7,No leading/trailing categorical whitespace remains,True,0,0
8,Inconsistent region labels removed,True,0,0
9,Misspelled security label removed,True,0,0


All post-cleaning validation checks passed.


In [7]:
before_after = pd.DataFrame({
    'metric': ['rows', 'columns', 'exact duplicates', 'missing target', 'default cases', 'non-default cases'],
    'raw': [
        len(raw), raw.shape[1], int(raw.duplicated().sum()), int(raw['Status'].isna().sum()),
        int(raw['Status'].eq(1).sum()), int(raw['Status'].eq(0).sum())
    ],
    'cleaned': [
        len(cleaned), cleaned.shape[1], int(cleaned.duplicated().sum()), int(cleaned['status'].isna().sum()),
        int(cleaned['status'].eq(1).sum()), int(cleaned['status'].eq(0).sum())
    ]
})
before_after['difference'] = before_after['cleaned'] - before_after['raw']
save_table(before_after, CLEANING_TABLES / 'before_after_summary.csv')
display(before_after)


,metric,raw,cleaned,difference
0,rows,148670,148670,0
1,columns,34,34,0
2,exact duplicates,0,0,0
3,missing target,0,0,0
4,default cases,36639,36639,0
5,non-default cases,112031,112031,0


## 6. Save the canonical cleaned dataset


In [8]:
CLEANED_DATA.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(CLEANED_DATA, index=False, encoding='utf-8')

digest = hashlib.sha256(CLEANED_DATA.read_bytes()).hexdigest()
export_manifest = pd.DataFrame({
    'field': ['output_path', 'rows', 'columns', 'sha256', 'source_file', 'model_training_performed'],
    'value': [str(CLEANED_DATA), len(cleaned), cleaned.shape[1], digest, str(RAW_DATA), False]
})
save_table(export_manifest, CLEANING_TABLES / 'cleaned_export_manifest.csv')
display(export_manifest)
print('Cleaning complete. No imputation, encoding, scaling, splitting, training, or testing was run.')


,field,value
0,output_path,/Users/zichen/Machine_Learning/data/processed/loan_default_cleaned.csv
1,rows,148670
2,columns,34
3,sha256,c1975912253142ef42ed15301e2ace013010a1d2ee1e06dda94fbd7f525b01ff
4,source_file,/Users/zichen/Machine_Learning/data/raw/Loan_Default.csv
5,model_training_performed,False


Cleaning complete. No imputation, encoding, scaling, splitting, training, or testing was run.
